# FloorPlanCAD Object Detection — Colab Training
**Trước khi chạy**: `Runtime` → `Change runtime type` → **GPU (T4 hoặc A100)**

### Flow đơn giản
```
Drive/MyDrive/Research/multi_model/object_detection/
  data/
    FloorPlanCAD_original/   ← zips + extracted PNGs + SVGs + *_meta.json
    zips/                    ← zip files (download 1 lần)
  checkpoints/               ← model weights
```
Không cần copy ảnh, không tar/untar — đọc thẳng từ Drive.


## 1 — Setup


In [ ]:
# Kiểm tra GPU
import torch
print(f'GPU  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — check runtime type!"}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR  = '/content/drive/MyDrive/Research/multi_model/object_detection'
ORIG_DIR  = f'{WORK_DIR}/data/FloorPlanCAD_original'
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
os.makedirs(ORIG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Data  : {ORIG_DIR}')
print(f'Ckpts : {CKPT_DIR}')

# Clone / update code
%cd /content
!git clone https://github.com/DaniYLab/object_detection.git 2>/dev/null || (cd object_detection && git pull origin main)
%cd /content/object_detection

# Symlink data vào repo (code đọc './data/...')
!rm -f data
import subprocess
subprocess.run(['ln', '-sf', f'{WORK_DIR}/data', 'data'], check=True)
!ls data/


## 2 — Install dependencies


In [ ]:
# Colab đã có torch CUDA — chỉ cài thêm thư viện
!pip install -q gdown Pillow tqdm svgpathtools matplotlib

import torch
print(f'torch : {torch.__version__}')
print(f'cuda  : {torch.version.cuda}')


## 3A — LẦN ĐẦU: Download zip gốc về Drive
> ⏭️ Skip nếu `Drive/.../data/FloorPlanCAD_original/` đã có PNGs rồi


In [ ]:
# Download 3 zips thẳng vào Drive, tự extract
!python scripts/data/download_gdrive.py --output_dir {ORIG_DIR}


## 3B — LẦN ĐẦU: Generate metadata
> ⏭️ Skip nếu đã có `*_meta.json` cạnh mỗi PNG rồi
>
> Chỉ tạo `*_meta.json`, không copy ảnh — chạy 1 lần duy nhất.


In [ ]:
# Parse SVG → tạo *_meta.json cạnh mỗi PNG
# Kết quả lưu thẳng trên Drive, không cần tar/untar
!python scripts/data/build_dataset.py --data_root {ORIG_DIR}


## 4 — Verify dataset + model


In [ ]:
import sys
sys.path.insert(0, '/content/object_detection')

from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES
from src.models.detector import FloorPlanDetector

# dataset đọc thẳng từ ORIG_DIR (qua symlink data/)
train_ds = FloorPlanDataset('./data/FloorPlanCAD_original', split='train', image_size=512)
val_ds   = FloorPlanDataset('./data/FloorPlanCAD_original', split='test',  image_size=512)
print(f'Train : {len(train_ds):,} samples')
print(f'Val   : {len(val_ds):,} samples')
print(f'Classes: {NUM_CLASSES}')

s = train_ds[0]
print(f'Image         : {s["image"].shape}')
print(f'center_heatmap: {s["center_heatmap"].shape}')
print(f'text          : {s["text"]}')
print(f'class_id      : {s["class_id"]} ({CLASS_NAMES[s["class_id"]]})')

model = FloorPlanDetector(image_size=512, model_dim=256, num_classes=NUM_CLASSES, depth_per_class=2)
n = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\nModel params: {n:.1f}M')


## 5 — Training

| GPU | VRAM | batch_size |
|-----|------|------------|
| T4  | 16GB | 4          |
| L4  | 24GB | 8          |
| A100| 40GB | 16         |


In [ ]:
import os
os.makedirs(CKPT_DIR, exist_ok=True)

!python train.py \
    --data_root    ./data/FloorPlanCAD_original \
    --ckpt_dir     {CKPT_DIR} \
    --image_size   512 \
    --batch_size   4 \
    --num_workers  2 \
    --model_dim    256 \
    --depth_per_class 2 \
    --epochs       50 \
    --lr           1e-4 \
    --log_interval 50


## 6 — Load checkpoint & Visualize


In [ ]:
import torch, matplotlib.pyplot as plt
from src.models.detector import FloorPlanDetector
from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES

ckpt  = torch.load(f'{CKPT_DIR}/best.pt', map_location='cpu')
model = FloorPlanDetector(image_size=512, model_dim=256, num_classes=NUM_CLASSES, depth_per_class=2)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Epoch {ckpt["epoch"]} | Val loss: {ckpt["val_loss"]:.4f}')

ds = FloorPlanDataset('./data/FloorPlanCAD_original', split='test', image_size=512)
sample = ds[0]

img = sample['image'].unsqueeze(0)
with torch.no_grad():
    preds = model(img)  # class_ids=None → all 35 classes

hm = preds['center_heatmap'][0]  # [35, H, W]
class_scores = hm.flatten(1).max(dim=-1).values
top5 = class_scores.topk(5)
for score, idx in zip(top5.values, top5.indices):
    print(f'  {CLASS_NAMES[idx]:20s} max={score:.3f}')

best_cid = top5.indices[0].item()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
img_np = (sample['image'].permute(1,2,0).numpy() * 0.5 + 0.5).clip(0, 1)
axes[0].imshow(img_np); axes[0].set_title('Original')
axes[1].imshow(hm[best_cid].numpy(), cmap='hot')
axes[1].set_title(f'{CLASS_NAMES[best_cid]} heatmap')
plt.tight_layout(); plt.show()
